In [0]:
from pyspark.sql.functions import *

orders = spark.table("ecommerce_dev.silver.orders")
dim_customer = spark.table("ecommerce_dev.gold.dim_customer")
dim_date = spark.table("ecommerce_dev.gold.dim_date")
order_payments = spark.table("ecommerce_dev.silver.order_payments")

In [0]:
fact_payments = (
    order_payments
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "order_id",
        "payment_sequential",
        "customer_key",
        "order_date_key",
        "payment_type",
        "payment_installments",
        "payment_value",
        "has_invalid_installments"
    )
)

print("Fact rows:", fact_payments.count())
print("Null customer_key:", fact_payments.filter(col("customer_key").isNull()).count())
print("Null order_date_key:", fact_payments.filter(col("order_date_key").isNull()).count())

Fact rows: 103886
Null customer_key: 0
Null order_date_key: 0


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.fact_payments (
    fact_payment_key BIGINT GENERATED ALWAYS AS IDENTITY,
    order_id STRING,
    payment_sequential INT,
    customer_key BIGINT,
    order_date_key INT,
    payment_type STRING,
    payment_installments INT,
    payment_value DECIMAL(10,2),
    has_invalid_installments BOOLEAN
) USING DELTA
""")

fact_payments.write.format("delta").mode("append").saveAsTable("ecommerce_dev.gold.fact_payments")

spark.sql("ALTER TABLE ecommerce_dev.gold.fact_payments ALTER COLUMN fact_payment_key SET NOT NULL")
spark.sql("ALTER TABLE ecommerce_dev.gold.fact_payments ADD CONSTRAINT pk_fact_payments PRIMARY KEY (fact_payment_key)")

spark.sql("""
COMMENT ON TABLE ecommerce_dev.gold.fact_payments IS
'Payment fact table at order_id + payment_sequential grain. FKs to Dim_Customer, Dim_Date. has_invalid_installments carried from Silver per flag-over-drop policy.'
""")

DataFrame[]